# 02 — Train teacher on H100 (Qwen2.5-14B-Instruct + QLoRA)

**Goal:** the largest model + best post-training that fits a single H100 (80GB) cleanly.

**Choices and *why*:**
- **Qwen2.5-14B-Instruct** — fits H100 in 4-bit with room for batch, optimizer state, and long ARC contexts. (32B is possible with QLoRA but inference becomes painful and TTT in notebook 04 needs headroom.)
- **QLoRA (4-bit base + LoRA on attention + MLP)** — keeps memory low, trains fast, and produces a small adapter that's cheap to swap and distill.
- **Training data:** ARC-AGI-1 train (400) + RE-ARC (Hodel's procedurally generated re-examples) + BARC (Berkeley synthetic, ~400K tasks generated by GPT-4). BARC is the biggest single source of signal in public ARC training.
- **Augmentations applied online:** 8 dihedral symmetries (rot/reflect) + random color permutations + example shuffling. Each base task becomes ~50–100 distinct training samples.
- **Loss:** standard SFT cross-entropy on the test-output tokens only (mask the prompt). This is what every working public ARC fine-tune does.
- **No DPO/RL in v1.** Keep it simple. Once SFT works, DPO can be added with rejected = sampled-incorrect-completion, chosen = ground-truth.

Run on Colab Pro+ with H100, ~6–10 hours for 1 epoch over ~50k augmented samples.


In [ ]:
# Hardware check — confirm H100
import torch
assert torch.cuda.is_available(), "No GPU detected."
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} ({vram:.0f} GB)")
if "H100" not in name and vram < 70:
    print("⚠ Not an H100 — you may need to drop to Qwen2.5-7B (set MODEL_ID below).")


In [ ]:
# Dependencies — pinned versions known to work together
!pip -q install -U transformers==4.46.2 accelerate==1.1.1 bitsandbytes==0.44.1 \
    datasets==3.1.0 peft==0.13.2 trl==0.12.1 huggingface_hub==0.26.2 sentencepiece


In [ ]:
import os, json, random, copy
from pathlib import Path
import numpy as np
import torch

random.seed(0); np.random.seed(0); torch.manual_seed(0)

DATA_DIR = Path("/content/arc_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR  = Path("/content/teacher_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 1) ARC-AGI-1 train + eval
import urllib.request, zipfile, io
if not (DATA_DIR / "training").exists():
    url = "https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip"
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
        z.extractall(DATA_DIR)
    src = DATA_DIR / "ARC-AGI-master" / "data"
    for kind in ["training", "evaluation"]:
        (DATA_DIR / kind).mkdir(exist_ok=True)
        for f in (src / kind).glob("*.json"):
            (DATA_DIR / kind / f.name).write_bytes(f.read_bytes())

arc_train = [json.loads(p.read_text()) for p in sorted((DATA_DIR / "training").glob("*.json"))]
arc_eval  = [json.loads(p.read_text()) for p in sorted((DATA_DIR / "evaluation").glob("*.json"))]
print(f"ARC: {len(arc_train)} train, {len(arc_eval)} eval")


In [ ]:
# 2) RE-ARC — Hodel's procedural re-generations of ARC tasks
# Repo: https://github.com/michaelhodel/re-arc — provides 1000s of variants per task.
# We download a precomputed bundle (small enough to fit in /content).
RE_ARC_URL = "https://github.com/michaelhodel/re-arc/raw/main/re_arc.zip"
re_arc_dir = DATA_DIR / "re_arc"
if not re_arc_dir.exists():
    try:
        with urllib.request.urlopen(RE_ARC_URL) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(re_arc_dir)
        print("RE-ARC downloaded")
    except Exception as e:
        print("RE-ARC download failed (skipping):", e)

re_arc_tasks = []
if re_arc_dir.exists():
    for fp in sorted(re_arc_dir.rglob("*.json"))[:5000]:  # cap for memory
        try:
            re_arc_tasks.append(json.loads(fp.read_text()))
        except Exception:
            pass
print(f"RE-ARC tasks loaded: {len(re_arc_tasks)}")


In [ ]:
# 3) BARC — Berkeley synthetic ARC tasks (~400k via GPT-4)
# Hosted on HuggingFace: barc-llm/barc-200k-gp4-induction-gpt4-description
# This is the single biggest signal source for public ARC fine-tunes.
from datasets import load_dataset
try:
    barc = load_dataset("barc0/200k_HEAVY_gpt4o-description-gpt4omini-code_generated_problems", split="train")
    print("BARC loaded:", len(barc), "tasks")
except Exception as e:
    print("BARC load failed (continuing without):", e)
    barc = None


## Augmentations

ARC problems are *invariant* under the dihedral group D4 (8 symmetries: rotations + reflections) and under color permutations (since color is just a categorical label, not a value with semantics). We exploit this to multiply our data ~8 × 10 = 80x.

This is the single most important data trick for ARC training.


In [ ]:
# Geometric augmentations (D4 symmetries)
def rot90(g):  return [list(row) for row in zip(*g[::-1])]
def fliplr(g): return [row[::-1] for row in g]
def transp(g): return [list(row) for row in zip(*g)]

D4 = [
    lambda g: g,
    rot90,
    lambda g: rot90(rot90(g)),
    lambda g: rot90(rot90(rot90(g))),
    fliplr,
    lambda g: rot90(fliplr(g)),
    lambda g: rot90(rot90(fliplr(g))),
    lambda g: rot90(rot90(rot90(fliplr(g)))),
]

def apply_d4(task, k):
    f = D4[k]
    return {
        "train": [{"input": f(ex["input"]), "output": f(ex["output"])} for ex in task["train"]],
        "test":  [{"input": f(ex["input"]), "output": f(ex["output"])} for ex in task["test"]],
    }

def apply_color_perm(task, perm):
    # perm: list of length 10 mapping color i -> perm[i]
    def remap(g):
        return [[perm[c] for c in row] for row in g]
    return {
        "train": [{"input": remap(ex["input"]), "output": remap(ex["output"])} for ex in task["train"]],
        "test":  [{"input": remap(ex["input"]), "output": remap(ex["output"])} for ex in task["test"]],
    }

def shuffle_examples(task, rng):
    t = copy.deepcopy(task)
    rng.shuffle(t["train"])
    return t


In [ ]:
# Prompt formatting — same as baseline so train/eval match
def grid_to_str(g):
    return "\n".join(" ".join(str(c) for c in row) for row in g)

SYSTEM = ("You are an expert ARC puzzle solver. Output only the test output grid as digits "
          "separated by spaces, one row per line. No explanations.")

def format_user(task):
    parts = ["You are solving an ARC puzzle. Each grid is a 2D array of digits 0-9 (colors).",
             "Find the transformation rule from the input/output examples, then apply it to the test input.",
             ""]
    for i, ex in enumerate(task["train"], 1):
        parts.append(f"Example {i} Input:");  parts.append(grid_to_str(ex["input"]))
        parts.append(f"Example {i} Output:"); parts.append(grid_to_str(ex["output"]))
        parts.append("")
    parts.append("Test Input:");  parts.append(grid_to_str(task["test"][0]["input"]))
    parts.append(""); parts.append("Test Output:")
    return "\n".join(parts)

def format_assistant(task):
    return grid_to_str(task["test"][0]["output"])


In [ ]:
# Build the augmented training set as raw (prompt, completion) pairs
# Token-budget: cap at ~3500 tokens to keep batches sane on H100.

def gen_augmented(base_tasks, n_per_task=20, rng=None):
    rng = rng or random.Random(0)
    out = []
    for task in base_tasks:
        for _ in range(n_per_task):
            t = task
            t = apply_d4(t, rng.randrange(8))
            perm = list(range(10)); rng.shuffle(perm)
            t = apply_color_perm(t, perm)
            t = shuffle_examples(t, rng)
            out.append(t)
    return out

def _coerce_examples(exs):
    """Each ex -> {'input': 2D-list, 'output': 2D-list}. Handles dict, list[2], tuple[2]."""
    if not isinstance(exs, list): return None
    out = []
    for e in exs:
        if isinstance(e, dict) and "input" in e and "output" in e:
            inp, outp = e["input"], e["output"]
        elif isinstance(e, (list, tuple)) and len(e) == 2:
            inp, outp = e[0], e[1]
        else:
            return None
        if not (isinstance(inp, list) and inp and isinstance(inp[0], list)): return None
        if not (isinstance(outp, list) and outp and isinstance(outp[0], list)): return None
        out.append({"input": inp, "output": outp})
    return out

def normalize_barc(row):
    train, test = None, None
    if "train" in row and "test" in row:
        train = _coerce_examples(row["train"]); test = _coerce_examples(row["test"])
    elif "examples" in row and "test" in row:
        train = _coerce_examples(row["examples"]); test = _coerce_examples(row["test"])
    elif "examples" in row:
        ex = _coerce_examples(row["examples"])
        if ex and len(ex) >= 2: train, test = ex[:-1], [ex[-1]]
    return {"train": train, "test": test} if (train and test) else None

base = list(arc_train) + list(re_arc_tasks)
if barc is not None:
    bsel = barc.select(range(min(20000, len(barc))))  # cap
    for r in bsel:
        norm = normalize_barc(r)
        if norm and norm["train"] and norm["test"]:
            base.append(norm)

print(f"Base task pool: {len(base)}")
augmented = gen_augmented(base, n_per_task=4)  # tune higher if epoch budget allows
print(f"Augmented samples: {len(augmented)}")


In [ ]:
# Tokenizer + length filter (drop samples longer than 3500 tokens to keep training fast)
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"  # downgrade to 7B if not on H100
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

MAX_LEN = 3500

def to_chat(t):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": format_user(t)},
        {"role": "assistant", "content": format_assistant(t)},
    ]
    return tok.apply_chat_template(msgs, tokenize=False)

texts = []
for t in augmented:
    s = to_chat(t)
    if len(tok(s).input_ids) <= MAX_LEN:
        texts.append(s)
print(f"After length filter: {len(texts)} samples")


In [ ]:
# Build a HuggingFace Dataset and train with TRL's SFTTrainer
from datasets import Dataset
ds = Dataset.from_dict({"text": texts})
ds = ds.shuffle(seed=0)
print(ds)


In [ ]:
# Load model in 4-bit with LoRA adapters
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",  # H100 supports it
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
# Train with TRL SFTTrainer — completion-only loss
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=str(OUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch 16
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_seq_length=MAX_LEN,
    dataset_text_field="text",
    packing=False,                    # ARC samples are heterogeneous, packing hurts
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tok,
    train_dataset=ds,
    args=cfg,
)
trainer.train()


In [ ]:
# Save adapter (small, ~200MB) — this is what you'll download to distill or eval
trainer.model.save_pretrained(str(OUT_DIR / "adapter"))
tok.save_pretrained(str(OUT_DIR / "adapter"))
print("Saved teacher LoRA adapter to", OUT_DIR / "adapter")


In [ ]:
# (Optional) push adapter to HF Hub so notebooks 03 and 04 can pull it
# from huggingface_hub import login, HfApi
# login()  # paste a write token
# HfApi().upload_folder(folder_path=str(OUT_DIR / "adapter"),
#                       repo_id="<your-username>/qwen25-14b-arc-lora",
#                       repo_type="model", create_pr=False)


In [ ]:
# Quick sanity eval on first 20 ARC eval tasks (no TTT yet — that's notebook 04)
import json, torch

@torch.inference_mode()
def predict(task, max_new=2048):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":format_user(task)}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

def parse_grid(s):
    rows = []
    for line in s.strip().splitlines():
        line = line.strip()
        if not line: continue
        try: rows.append([int(c) for c in line.split()])
        except: return None
    if not rows or any(len(r)!=len(rows[0]) for r in rows): return None
    return rows

correct = 0
N = 20
for i, task in enumerate(arc_eval[:N]):
    raw = predict(task)
    if parse_grid(raw) == task["test"][0]["output"]:
        correct += 1
print(f"Quick eval: {correct}/{N} = {correct/N:.1%}")
